# 11 - Hyperparameter Tuning dengan Optuna

## Context
Model machine learning sering kali membutuhkan penyesuaian parameter (hyperparameter tuning) agar dapat bekerja secara optimal. Pada notebook ini, kita menggunakan Optuna untuk mencari kombinasi hyperparameter terbaik untuk model LightGBM, XGBoost, dan CatBoost. Optuna menggunakan pencarian berbasis Bayesian untuk menemukan parameter terbaik dengan cepat dan efisien.

## Objective
- Mengoptimalkan hyperparameter untuk LightGBM, XGBoost, dan CatBoost menggunakan Optuna.
- Menyimpan parameter terbaik untuk digunakan pada tahap evaluasi final dan ensemble.

### Impor Library

In [ ]:
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
optuna.logging.set_verbosity(optuna.logging.WARNING)

### Muat Dataset Terproses

In [ ]:
path = Path("../data/processed/train_features.parquet")
if not path.exists():
    path = Path("../data/interim/train_merged.parquet")
train = pd.read_parquet(path)
print(f"Data berhasil dimuat: {train.shape}")

### LightGBM Hyperparameter Tuning

In [ ]:
# Pisahkan fitur dan target
exclude_cols = ["TransactionID", "isFraud", "TransactionDT"]
feature_cols = [c for c in train.select_dtypes(include="number").columns if c not in exclude_cols]
X = train[feature_cols]
y = train["isFraud"]

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)

def objective_lgb(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "verbose": -1,
        "random_state": 42
    }
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=100,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(15, verbose=False)]
    )
    preds = model.predict(X_va, num_iteration=model.best_iteration)
    return roc_auc_score(y_va, preds)

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=5)
print(f"Skor LGBM Terbaik: {study_lgb.best_value:.5f}")
Path("../data/metadata").mkdir(parents=True, exist_ok=True)
with open("../data/metadata/lightgbm_best_params.json", "w") as f:
    json.dump(study_lgb.best_params, f, indent=1)

### XGBoost Hyperparameter Tuning

In [ ]:
dtrain_x = xgb.DMatrix(X_tr, label=y_tr)
dval_x = xgb.DMatrix(X_va, label=y_va)

def objective_xgb(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "verbosity": 0,
        "random_state": 42
    }
    early_stop = xgb.callback.EarlyStopping(rounds=15, save_best=True, metric_name="auc", data_name="val")
    model = xgb.train(
        params,
        dtrain_x,
        num_boost_round=100,
        evals=[(dval_x, "val")],
        callbacks=[early_stop],
        verbose_eval=False
    )
    preds = model.predict(dval_x)
    return roc_auc_score(y_va, preds)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=5)
print(f"Skor XGBoost Terbaik: {study_xgb.best_value:.5f}")

### CatBoost Hyperparameter Tuning

In [ ]:
def objective_cb(trial):
    params = {
        "iterations": 100,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "eval_metric": "AUC",
        "random_seed": 42,
        "verbose": False
    }
    model = CatBoostClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=15)
    preds = model.predict_proba(X_va)[:, 1]
    return roc_auc_score(y_va, preds)

study_cb = optuna.create_study(direction="maximize")
study_cb.optimize(objective_cb, n_trials=5)
print(f"Skor CatBoost Terbaik: {study_cb.best_value:.5f}")

### Perbandingan Hasil Tuning

In [ ]:
tuning_results = [
    {"Model": "LightGBM", "Best Val AUC": study_lgb.best_value, "Trials": len(study_lgb.trials)},
    {"Model": "XGBoost", "Best Val AUC": study_xgb.best_value, "Trials": len(study_xgb.trials)},
    {"Model": "CatBoost", "Best Val AUC": study_cb.best_value, "Trials": len(study_cb.trials)}
]
print(pd.DataFrame(tuning_results))